# i2Nav V2 Backward-Derivative LOSO Sensitivity

This notebook retrains both the V1 covariance stage and V2 correction stage for 10 folds x 3 seeds after replacing centered acceleration derivatives with backward-only derivatives. It tests whether the learned model and later ledger survive removal of the known one-sample derivative lookahead. It is a **new sensitivity pipeline**, not the original frozen V2 and not proof of fully causal sensor-to-service operation: interpolation availability and execution/transport latency remain separate.

In [ ]:

from pathlib import Path
import csv,hashlib,json,os,shutil,subprocess,sys,zipfile
REPO_URL='https://github.com/CEISCA-VT/DigitalTwinDivergence.git'; REPO_REF='main'; EXPECTED_COMMIT='6f91f5dae834ac55dfebf3abae76beba7379ff2c'
BASE_SEEDS=[42,1042,2042]; SHARD_INDEX=0; SHARD_COUNT=6; DEVICE='cuda'
WORK=Path('/kaggle/working'); REPO=WORK/'DigitalTwinDivergence'; OUTPUT=WORK/'i2nav_v2_causal_derivative_loso'; LOGS=WORK/'i2nav_v2_causal_derivative_logs'
OUTPUT.mkdir(parents=True,exist_ok=True); LOGS.mkdir(parents=True,exist_ok=True)
if not EXPECTED_COMMIT or EXPECTED_COMMIT == 'REPLACE_WITH_PUSHED_COMMIT' or len(EXPECTED_COMMIT) != 40: raise ValueError('Set EXPECTED_COMMIT to the full pushed repository commit before launching shards')
if not 0 <= SHARD_INDEX < SHARD_COUNT: raise ValueError('Invalid shard')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(REPO)],check=True)
COMMIT=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if COMMIT != EXPECTED_COMMIT: raise RuntimeError(f'Commit mismatch: {COMMIT} != {EXPECTED_COMMIT}')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'

def sha256_file(path, block_size=1024*1024):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(block_size), b''):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

print(COMMIT)


In [ ]:
probe="import json,torch; print(json.dumps({'ok':torch.cuda.is_available(),'cap':list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list() if torch.cuda.is_available() else []}))"
p=subprocess.run([sys.executable,'-c',probe],text=True,capture_output=True,check=True); gpu=json.loads(p.stdout.strip().splitlines()[-1]); print(gpu)
if not gpu['ok']: raise RuntimeError('Enable a Kaggle GPU')
if tuple(gpu['cap'])==(6,0) and 'sm_60' not in gpu['arch']:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--force-reinstall','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
real="import torch; x=torch.randn(64,64,device='cuda'); g=torch.nn.GRU(6,64,2,batch_first=True).cuda(); y,_=g(torch.randn(2,20,6,device='cuda')); torch.cuda.synchronize(); assert torch.isfinite(x@x.T).all() and torch.isfinite(y).all()"
subprocess.run([sys.executable,'-c',real],check=True); print('CUDA PREFLIGHT: PASS')


In [ ]:

v1_runner='DigitalTwin.analysis.i2nav_loso_ablation'; v2_runner='DigitalTwin.analysis.i2nav_v2_full_loso'
v1_help=subprocess.check_output([sys.executable,'-m',v1_runner,'--help'],cwd=REPO,text=True); v2_help=subprocess.check_output([sys.executable,'-m',v2_runner,'--help'],cwd=REPO,text=True)
if '--feature-derivative-mode' not in v1_help or '--v1-checkpoint' not in v2_help: raise RuntimeError('Repository commit lacks causal V1/V2 support')

# Numerical no-lookahead preflight. This checks every derivative-bearing feature from the helper, not only a manifest flag.
preflight = """
import json
import numpy as np
from DigitalTwin.analysis.i2nav_v2_full_loso import causal_backward_features
rng=np.random.default_rng(7)
grid=np.arange(0.0,3.0,0.1)
speed=rng.normal(size=len(grid)); omega=rng.normal(size=len(grid))
base=causal_backward_features(speed,omega,grid)
changed=[]
for future_idx in range(1,len(grid)):
    sp=speed.copy(); om=omega.copy(); sp[future_idx]+=123.0; om[future_idx]-=77.0
    alt=causal_backward_features(sp,om,grid)
    if not np.allclose(base[:future_idx],alt[:future_idx],equal_nan=True):
        changed.append(int(future_idx))
assert not changed, changed[:5]
expected_speed=np.r_[0.0,np.diff(speed)/np.diff(grid)]
expected_omega=np.r_[0.0,np.diff(omega)/np.diff(grid)]
assert np.allclose(base[:,2],expected_speed,equal_nan=True)
assert np.allclose(base[:,3],expected_omega,equal_nan=True)
print(json.dumps({'status':'PASS','feature_columns':['speed','omega','speed_accel_backward','omega_accel_backward','abs_omega','curvature_proxy'],'first_sample_boundary_rule':'acceleration columns are set to 0.0 at the first sample; all later derivative samples use k and k-1 only'}))
"""
p=subprocess.run([sys.executable,'-c',preflight],cwd=REPO,text=True,capture_output=True,check=True)
causality_preflight=json.loads(p.stdout.strip().splitlines()[-1])
(OUTPUT/'causal_no_lookahead_preflight.json').write_text(json.dumps({'schema':'causal_backward_no_lookahead_preflight_v1','commit':COMMIT,**causality_preflight},indent=2)+'\n')
print(causality_preflight)

seqs=['building00','building01','building02','parking00','parking01','parking02','playground00','street00','street01','street02']
tasks=[{'test':s,'seed':seed} for s in seqs for seed in BASE_SEEDS]; tasks=sorted(tasks,key=lambda x:(x['test'],x['seed']))
shard=[t for i,t in enumerate(tasks) if i%SHARD_COUNT==SHARD_INDEX]
ledger={'schema':'causal_derivative_v2_loso_ledger_v2','commit':COMMIT,'derivative_mode':'causal_backward','total_tasks':30,'shard_index':SHARD_INDEX,'shard_count':SHARD_COUNT,'tasks':shard,'causality_preflight':'causal_no_lookahead_preflight.json'}
(OUTPUT/f'task_ledger_shard_{SHARD_INDEX:02d}.json').write_text(json.dumps(ledger,indent=2)+'\n')
print(f'This shard: {len(shard)} of 30 paired V1-to-V2 pipelines')


In [ ]:

def v1_complete(v1_out, target):
    checkpoint=v1_out/'folds'/target/'gru_dual.pt'; results=v1_out/'loso_results.csv'; split=v1_out/'fold_splits.json'
    if not (checkpoint.is_file() and results.is_file() and split.is_file()): return False
    rows=read_json(split)
    if len(rows)!=1 or rows[0].get('test')!=target or rows[0].get('feature_derivative_mode')!='causal_backward': return False
    with results.open(newline='',encoding='utf-8') as f:
        return any(r.get('method')=='gru_dual' and r.get('status')=='ok' for r in csv.DictReader(f))

def v2_manifest_for(target, seed):
    found=[]
    for p in (OUTPUT/'v2').rglob(f'*_{target}/run_manifest.json'):
        try: m=read_json(p)
        except Exception: continue
        if m.get('base_seed')==seed: found.append((p,m))
    return found

def v2_complete(target, seed, expected_v1, expected_results):
    matches=v2_manifest_for(target, seed)
    if len(matches)!=1: return False
    p,m=matches[0]
    required=[p.with_name('RUN_COMPLETE.json'),p.with_name('run_summary.json'),p.with_name('v2_slow_additive_yaw.pt'),p.with_name('v2_evaluated_trajectory.csv'),p.with_name('v2_prediction_trace.csv'),p.with_name('fidelity_profile.json'),p.with_name('fidelity_timeseries.csv')]
    return all(x.is_file() for x in required) and m.get('status')=='complete' and m.get('feature_derivative_mode')=='causal_backward' and Path(m.get('v1_checkpoint_source','')).resolve()==expected_v1.resolve() and Path(m.get('v1_results_source','')).resolve()==expected_results.resolve()

env=os.environ.copy(); env['PYTHONPATH']=str(REPO)
for i,t in enumerate(shard,1):
    log=LOGS/f"test-{t['test']}__seed-{t['seed']}.log"; v1_out=OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']; v2_out=OUTPUT/'v2'
    v1_cmd=[sys.executable,'-m',v1_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--output-dir',str(v1_out),'--folds',t['test'],'--methods','gru_dual','--save-trajectories','--seed',str(t['seed']),'--device',DEVICE,'--feature-derivative-mode','causal_backward']
    v1_checkpoint=v1_out/'folds'/t['test']/'gru_dual.pt'; v1_results=v1_out/'loso_results.csv'
    v2_cmd=[sys.executable,'-m',v2_runner,'--root',str(REPO/'public_datasets'/'im2nav'),'--frozen-v1-dir',str(REPO/'results'/'i2nav_v1_frozen'),'--output-dir',str(v2_out),'--test-sequence',t['test'],'--base-seed',str(t['seed']),'--device',DEVICE,'--feature-derivative-mode','causal_backward','--v1-checkpoint',str(v1_checkpoint),'--v1-results-csv',str(v1_results)]
    print(f'[{i}/{len(shard)}] {t}',flush=True)
    if v1_complete(v1_out,t['test']):
        print('  V1 complete; skipping')
    else:
        with log.open('a',encoding='utf-8') as f:
            result=subprocess.run(v1_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
        if result.returncode: raise RuntimeError(f'V1 fit failed: {log}')
    if v2_complete(t['test'],t['seed'],v1_checkpoint,v1_results):
        print('  V2 complete; skipping')
        continue
    with log.open('a',encoding='utf-8') as f:
        result=subprocess.run(v2_cmd,cwd=REPO,env=env,text=True,stdout=f,stderr=subprocess.STDOUT)
    if result.returncode: raise RuntimeError(f'V2 fit failed: {log}')
print('SHARD EXECUTION COMPLETE')


In [ ]:

audited=[]
for t in shard:
    candidates=v2_manifest_for(t['test'],t['seed'])
    if len(candidates)!=1: raise RuntimeError(f'Manifest count {t}: {len(candidates)}')
    p,m=candidates[0]; v1_split_path=OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']/'fold_splits.json'; v1_split=read_json(v1_split_path)[0]
    if v1_split.get('feature_derivative_mode')!='causal_backward': raise RuntimeError(f'V1 was not causal-backward: {v1_split_path}')
    for key in ['normalization_fit_sequences','checkpoint_selection_sequences']:
        if key not in v1_split: raise RuntimeError(f'Missing V1 provenance field {key}: {v1_split_path}')
    expected_v1=(OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']/'folds'/t['test']/'gru_dual.pt').resolve()
    expected_results=(OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']/'loso_results.csv').resolve()
    if m.get('feature_derivative_mode')!='causal_backward' or Path(m.get('v1_checkpoint_source','')).resolve()!=expected_v1: raise RuntimeError(f'Wrong causal V1 checkpoint provenance: {p}')
    if Path(m.get('v1_results_source','')).resolve()!=expected_results: raise RuntimeError(f'Wrong causal V1 results provenance: {p}')
    if m.get('v1_checkpoint_sha256')!=sha256_file(expected_v1): raise RuntimeError(f'V1 checkpoint hash mismatch: {p}')
    if m.get('v1_results_sha256')!=sha256_file(expected_results): raise RuntimeError(f'V1 results hash mismatch: {p}')
    if t['test'] in set(m['training_names'])|set(m['validation_names']): raise RuntimeError(f'Test leakage: {p}')
    if set(m.get('normalization_fit_sequences',[]))!=set(m['training_names']): raise RuntimeError(f'V2 normalization provenance mismatch: {p}')
    if set(m.get('checkpoint_selection_sequences',[]))!=set(m['validation_names']): raise RuntimeError(f'V2 checkpoint-selection provenance mismatch: {p}')
    if not p.with_name('RUN_COMPLETE.json').is_file(): raise RuntimeError(f'Incomplete: {p}')
    artifacts=m.get('artifacts_sha256',{})
    if artifacts.get('evaluated_trajectory')!=sha256_file(p.with_name('v2_evaluated_trajectory.csv')): raise RuntimeError(f'Trajectory hash mismatch: {p}')
    audited.append({**t,'v1_split':str(v1_split_path.relative_to(OUTPUT)),'v1_checkpoint_sha256':m.get('v1_checkpoint_sha256'),'v1_results_sha256':m.get('v1_results_sha256'),'v2_trajectory_sha256':artifacts.get('evaluated_trajectory'),'v2_manifest':str(p.relative_to(OUTPUT)),'status':'PASS'})
(OUTPUT/f'shard_audit_{SHARD_INDEX:02d}.json').write_text(json.dumps({'schema':'causal_derivative_shard_audit_v2','commit':COMMIT,'rows':audited,'complete':len(audited)==len(shard)},indent=2)+'\n')
print(f'AUDIT PASS: {len(audited)}/{len(shard)}')


In [ ]:

archive=WORK/f"i2nav_v2_causal_derivative_shard_{SHARD_INDEX:02d}_of_{SHARD_COUNT:02d}.zip"
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    shard_files=[OUTPUT/'causal_no_lookahead_preflight.json', OUTPUT/f'task_ledger_shard_{SHARD_INDEX:02d}.json', OUTPUT/f'shard_audit_{SHARD_INDEX:02d}.json']
    for t in shard:
        v1_parent=OUTPUT/'v1'/f"seed_{t['seed']}"/t['test']
        for p in v1_parent.rglob('*'):
            if p.is_file(): shard_files.append(p)
        for p in (OUTPUT/'v2').rglob(f"*_{t['test']}/*"):
            if p.is_file():
                try:
                    m=read_json(p.parent/'run_manifest.json')
                    if m.get('base_seed')==t['seed']: shard_files.append(p)
                except Exception:
                    pass
        log=LOGS/f"test-{t['test']}__seed-{t['seed']}.log"
        if log.is_file(): shard_files.append(log)
    for p in sorted(set(shard_files)):
        if p.is_file(): z.write(p,p.relative_to(WORK))
with zipfile.ZipFile(archive,'r') as z:
    bad=z.testzip()
    if bad: raise RuntimeError(f'ZIP integrity check failed at {bad}')
print({'archive':str(archive),'sha256':sha256_file(archive),'bytes':archive.stat().st_size})


## Interpretation boundary
Merge all shards before comparing this model with frozen V2. Report it as a backward-derivative sensitivity. The cheaper primary check remains adding the known feature-availability delay to the original precomputed-state replay; this notebook answers the stronger optional question of what happens after retraining the model with backward-only derivative inputs.